# RAG with a Local LLM

## Google Colab edition

**Applied AI Summer Workshop.**

The companion to the prompting notebook. That one covered controlling a
model's behavior; this one gives the model **access to documents it was never
trained on**.

That's **Retrieval-Augmented Generation (RAG)**: before the model answers, we
search a document collection for relevant passages and paste them into the
prompt as context. The model doesn't learn anything new — it just gets handed
the right page at the right moment.

| Part | Topic |
| --- | --- |
| **A** | Setup — Ollama, a chat model, and an embedding model |
| **B** | The corpus — documents the model wasn't trained on |
| **C** | Chunking — cutting documents into searchable pieces |
| **D** | Embeddings — turning text into vectors |
| **E** | Retrieval — finding the passages that matter |
| **F** | Generation — does retrieval actually change the answer? |

## ⚠️ This is not private

As with the prompting notebook: this runs on a **Google virtual machine**, so
your prompts and documents leave your computer. On the Hub or your own laptop
the same code is genuinely local. Worth keeping straight — especially here,
where RAG is often pitched as the way to query *sensitive internal documents*.
If the documents are the sensitive part, Colab is the wrong venue.

## Before you start: turn on the GPU

**Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**

# Part A — Setup

Run these **in order**. About 5 minutes, mostly downloading — this notebook
needs *two* kinds of model, so there's more to fetch than last time.

Re-run all of Part A whenever Colab gives you a fresh session.

### A1. Install [Ollama](https://ollama.com)

Ollama's installer unpacks a `.tar.zst` archive and Colab's image doesn't
ship `zstd`, so we install that first or the install fails.

In [ ]:
# Ollama's installer needs zstd, and Colab doesn't ship it.
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd

In [ ]:
# Takes about a minute. The output is noisy - that's normal.
!curl -fsSL https://ollama.com/install.sh | sh

### A2. Start the Ollama server

Started with `subprocess.Popen` rather than `!ollama serve`, because a `!`
command would block this cell forever.

In [ ]:
import subprocess
import time
import urllib.request

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)


def ollama_is_running():
    """Return True once the server is accepting connections."""
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False


for attempt in range(30):
    if ollama_is_running():
        print("Ollama server is up.")
        break
    time.sleep(1)
else:
    print("Server didn't start in 30s. Try re-running this cell.")

### A3. Download the models

RAG needs **two different kinds of model**, and the distinction matters:

- **a chat model** (`llama3.2:3b`) — writes the final answer.
- **embedding models** (`nomic-embed-text`, `mxbai-embed-large`) — turn text
  into vectors so we can search it. They don't generate text at all.

We pull two embedding models so Part D can compare them. Roughly 3 GB total,
so this is the slow cell.

In [ ]:
!ollama pull llama3.2:3b
!ollama pull nomic-embed-text
!ollama pull mxbai-embed-large

### A4. Check everything is working

In [ ]:
import json
import urllib.request

NEEDED = ["llama3.2:3b", "nomic-embed-text", "mxbai-embed-large"]

try:
    with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=5) as r:
        installed = [m["name"] for m in json.loads(r.read()).get("models", [])]
except Exception:
    installed = None

if installed is None:
    print("[X] Can't reach the Ollama server. Re-run cell A2.")
else:
    missing = [n for n in NEEDED if not any(m.startswith(n) for m in installed)]
    if missing:
        print(f"[X] Server is up, but these are missing: {missing}")
        print("    Re-run cell A3.")
    else:
        print("[OK] Server running; all three models present.")
        print(f"     {', '.join(installed)}")
        print("\nSetup complete - continue to Part B.")

# Part B — The corpus

RAG needs documents. Ours is a small set of Wikipedia articles on 19th-century
Russian literature, standing in for "your documents" — a paper archive, a
policy manual, lab notebooks, whatever you actually want to query.

On the Hub these files just sit in a `corpus/` folder. Colab starts empty, so
we have to get them here. **Run B.1 first**; only use B.2 if B.1 fails.

## B.1. Fetch the articles from Wikipedia

These are the same 12 articles as the Hub version, pulled live from
Wikipedia's API as plain text.

In [ ]:
import json
import os
import urllib.parse
import urllib.request

# The same 12 articles used by the Hub version of this demo.
ARTICLES = [
    "Anton Chekhov",
    "Crime and Punishment",
    "Dead Souls",
    "Fathers and Sons (novel)",
    "Fyodor Dostoevsky",
    "Ivan Turgenev",
    "Leo Tolstoy",
    "Nikolai Gogol",
    "Realism (arts)",
    "Russian literature",
    "The Brothers Karamazov",
    "War and Peace",
]

os.makedirs("corpus", exist_ok=True)


def fetch_wikipedia(title):
    """Return the plain-text extract of one Wikipedia article."""
    params = urllib.parse.urlencode({
        "action": "query",
        "prop": "extracts",
        "explaintext": "1",
        "format": "json",
        "titles": title,
        "redirects": "1",
    })
    url = f"https://en.wikipedia.org/w/api.php?{params}"
    request = urllib.request.Request(url, headers={"User-Agent": "WorkshopRAGDemo/1.0"})
    with urllib.request.urlopen(request, timeout=30) as response:
        pages = json.loads(response.read().decode("utf-8"))["query"]["pages"]
    return next(iter(pages.values())).get("extract", "")


fetched, failed = 0, []
for title in ARTICLES:
    try:
        text = fetch_wikipedia(title)
        # A real article is many KB; anything tiny means something went wrong.
        if len(text) < 2000:
            failed.append(title)
            continue
        filename = title.replace(" ", "_").replace("(", "").replace(")", "")
        with open(f"corpus/{filename}.md", "w", encoding="utf-8") as f:
            f.write(f"# {title}\n\n{text}")
        fetched += 1
    except Exception as error:
        failed.append(f"{title} ({type(error).__name__})")

print(f"Fetched {fetched} of {len(ARTICLES)} articles into corpus/")
if failed:
    print(f"Failed: {failed}")
if fetched >= 8:
    print("\n[OK] Enough documents to continue. Skip B.2 and go to Part C.")
else:
    print("\n[X] Not enough documents - use B.2 below to upload them instead.")

## B.2. Fallback — upload `corpus.zip` by hand

Only needed if B.1 failed (no network, Wikipedia blocked, API changed).

`corpus.zip` ships alongside this notebook in the workshop repository. Run the
cell, choose the file, and it unpacks into `corpus/`.

In [ ]:
# Only run this if B.1 didn't work.
from google.colab import files
import zipfile

uploaded = files.upload()  # pick corpus.zip in the file dialog

for filename in uploaded:
    if filename.endswith(".zip"):
        with zipfile.ZipFile(filename) as archive:
            archive.extractall(".")
        print(f"Unpacked {filename}")

import os
if os.path.isdir("corpus"):
    print(f"corpus/ now holds {len(os.listdir('corpus'))} files.")
else:
    print("[X] No corpus/ folder - check that you uploaded corpus.zip.")

## B.3. Load the documents

Either route leaves us with a `corpus/` folder. This reads it into memory.

In [ ]:
import os


def load_corpus(corpus_dir="corpus"):
    """Read every .md file in the folder into a {name: text} dict."""
    docs = {}
    for filename in sorted(os.listdir(corpus_dir)):
        if filename.endswith(".md"):
            with open(os.path.join(corpus_dir, filename), encoding="utf-8") as f:
                docs[filename[:-3]] = f.read()
    return docs


corpus = load_corpus()
total_words = sum(len(t.split()) for t in corpus.values())

print(f"Loaded {len(corpus)} documents, ~{total_words:,} words total:\n")
for name, text in corpus.items():
    print(f"  {name:<28} {len(text.split()):>7,} words")

# Part C — Chunking

A whole article is too long to hand the model as context, and mostly
irrelevant to any single question. So each document is split into smaller
**chunks**, and we search over those instead of whole articles.

Two rules, fixed here on purpose so there's one less knob to think about:

- split on `==` section headings, so each chunk is one topical section
- cap each chunk at **200 words**, splitting further if a section runs long

Chunk size is a real trade-off: **too small** and a passage loses the context
that made it meaningful; **too big** and the useful sentence gets diluted by
surrounding text, and you burn the model's limited context window.

In [ ]:
import re


def chunk_document(doc_id, text, max_words=200):
    """Split one document into a list of {doc_id, text} chunks."""
    # Two heading styles, because the corpus can arrive two ways:
    #   B.1 (Wikipedia API plain text) marks sections as  == Heading ==
    #   B.2 (uploaded corpus.zip markdown) marks them as  ## Heading
    # Handling both means section splitting works either way.
    sections = re.split(r"\n(?:==+ .+? ==+|#{2,} .+?)\n", text)

    chunks = []
    for section in sections:
        words = section.split()
        # Walk the section in max_words-sized windows.
        for start in range(0, len(words), max_words):
            piece = " ".join(words[start:start + max_words]).strip()
            if piece:
                chunks.append({"doc_id": doc_id, "text": piece})
    return chunks


chunks = []
for doc_id, text in corpus.items():
    chunks.extend(chunk_document(doc_id, text))

print(f"{len(chunks)} chunks from {len(corpus)} documents.\n")
print("Example chunk:")
print("-" * 60)
print(f"[{chunks[1]['doc_id']}]")
print(chunks[1]["text"][:400], "...")
print("-" * 60)

# Part D — Embeddings

To search the chunks, each one is turned into a vector — an **embedding**.
Text that means similar things lands in similar places in that vector space,
which is what lets us search by *meaning* rather than by keyword.

Three choices decide what "similar" ends up meaning:

**1. Which model.** `nomic-embed-text` and `mxbai-embed-large` are both served
by Ollama, but they're different models, with different training data and
different vector sizes.

**2. Instruction prefixes.** ⚠️ This is the subtle one. Many embedding models
were trained with a short prefix prepended to the text — *one for documents,
a different one for search queries*. Get this wrong and retrieval quietly gets
worse. No error, no warning, just worse results. The prefixes are written out
explicitly below rather than hidden in a config file, because getting them
right is the whole point.

**3. Similarity metric.** We L2-normalize every vector, which makes a plain
dot product between two vectors *equal* to their cosine similarity. That's the
entire search step: one matrix multiply.

In [ ]:
import json
import urllib.error
import urllib.request

import numpy as np

OLLAMA_URL = "http://127.0.0.1:11434"

# The prefix convention for each model - spelled out here, not buried in a
# registry, because getting this right (or wrong) is the point.
EMBED_MODELS = {
    "nomic-embed-text": {
        "doc_prefix": "search_document: ",
        "query_prefix": "search_query: ",
    },
    "mxbai-embed-large": {
        "doc_prefix": "",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
    },
}


def embed(texts, model):
    """POST a batch of strings to /api/embed, return L2-normalized vectors."""
    payload = {"model": model, "input": texts}
    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        f"{OLLAMA_URL}/api/embed", data=data,
        headers={"Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(request, timeout=300) as response:
            body = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        # HTTPError subclasses URLError, so catch it FIRST - otherwise
        # Ollama's actual message (bad model name, not pulled, etc.) gets
        # hidden behind a generic "can't reach the server".
        detail = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Ollama rejected the embed request for '{model}' "
            f"(HTTP {error.code}): {detail}\n"
            f"Has it been pulled? Try: ollama pull {model}"
        ) from error
    except urllib.error.URLError as error:
        raise RuntimeError(
            f"Could not reach Ollama at {OLLAMA_URL}. Re-run cell A2."
        ) from error

    vectors = np.array(body["embeddings"], dtype=np.float32)

    # L2-normalize so that a dot product IS cosine similarity.
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return vectors / norms


print("embed() defined - used for both chunks and questions.")

## D.1. Embed every chunk

**Change `EMBED_MODEL` below and re-run** to switch models. Part F revisits
this — different embedding models retrieve different passages, which can
change the final answer.

This is the slowest cell in the notebook: it embeds every chunk.

In [ ]:
import time

# <-- CHANGE THIS to "mxbai-embed-large" to compare.
EMBED_MODEL = "nomic-embed-text"

doc_prefix = EMBED_MODELS[EMBED_MODEL]["doc_prefix"]

# Note the document prefix on every chunk - NOT the query prefix.
texts = [f"{doc_prefix}{chunk['text']}" for chunk in chunks]

start_time = time.time()
chunk_vectors = embed(texts, EMBED_MODEL)
elapsed = time.time() - start_time

print(f"Embedded {chunk_vectors.shape[0]} chunks with {EMBED_MODEL}")
print(f"  took            {elapsed:.1f}s")
print(f"  vector dimension {chunk_vectors.shape[1]}")
print(f"  document prefix  {doc_prefix!r}")

# Part E — Retrieval

Now the search. The question gets embedded too — with its **query prefix**,
not the document prefix — and compared against every chunk vector with a dot
product. Highest score wins.

That's it. No vector database, no index: for a few hundred chunks, one matrix
multiply against everything is instant.

In [ ]:
def retrieve(question, k=4):
    """Return the k chunks most similar to the question."""
    # The QUERY prefix here - this is the asymmetry that's easy to get wrong.
    query_prefix = EMBED_MODELS[EMBED_MODEL]["query_prefix"]
    question_vector = embed([f"{query_prefix}{question}"], EMBED_MODEL)[0]

    # Both sides are L2-normalized, so this dot product is cosine similarity.
    scores = chunk_vectors @ question_vector

    top_indices = np.argsort(-scores)[:k]
    return [{"score": float(scores[i]), **chunks[i]} for i in top_indices]


# <-- CHANGE THIS to ask something else.
QUESTION = "Who challenged whom to a duel - Tolstoy or Turgenev?"

hits = retrieve(QUESTION)

print(f"Top {len(hits)} chunks for: {QUESTION}\n")
for rank, hit in enumerate(hits, 1):
    print(f"{rank}. score {hit['score']:.4f}  [{hit['doc_id']}]")
    print(f"   {hit['text'][:220]}...\n")

### Try the prefix mistake on purpose

Worth seeing once: use the **document** prefix on the question instead of the
query prefix — the mistake described in Part D — and watch the scores and the
retrieved passages change.

Nothing errors. It just quietly gets worse. That's what makes it dangerous.

In [ ]:
def retrieve_wrong_prefix(question, k=4):
    """Deliberately WRONG: uses the document prefix for the query."""
    wrong_prefix = EMBED_MODELS[EMBED_MODEL]["doc_prefix"]
    question_vector = embed([f"{wrong_prefix}{question}"], EMBED_MODEL)[0]
    scores = chunk_vectors @ question_vector
    top_indices = np.argsort(-scores)[:k]
    return [{"score": float(scores[i]), **chunks[i]} for i in top_indices]


right = retrieve(QUESTION)
wrong = retrieve_wrong_prefix(QUESTION)

print(f"Question: {QUESTION}\n")
print(f"{'CORRECT query prefix':<38} | {'WRONG (document) prefix':<38}")
print("-" * 79)
for r, w in zip(right, wrong):
    left = f"{r['score']:.3f} {r['doc_id'][:30]}"
    right_col = f"{w['score']:.3f} {w['doc_id'][:30]}"
    print(f"{left:<38} | {right_col:<38}")

same = [r["text"] for r in right] == [w["text"] for w in wrong]
print(f"\nRetrieved the same passages? {'yes' if same else 'NO - different results'}")
if not same:
    print("Same question, same model, same corpus. Only the prefix changed.")

# Part F — Generation: does retrieval actually change the answer?

The payoff. We ask the model the same question twice:

- **RAG off** — just the question. The model answers from memory.
- **RAG on** — the question *plus* the retrieved passages as context.

Same model, same question. Only the prompt differs.

In [ ]:
import json
import urllib.request


def call_ollama_chat(prompt, model="llama3.2:3b"):
    """POST to /api/generate and return the model's plain-text reply."""
    payload = {"model": model, "prompt": prompt, "stream": False}
    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        f"{OLLAMA_URL}/api/generate", data=data,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=180) as response:
        return json.loads(response.read().decode("utf-8"))["response"]


def answer(question, use_rag=True, k=4):
    """Answer a question, optionally grounding it in retrieved context."""
    if not use_rag:
        return call_ollama_chat(question), None

    hits = retrieve(question, k=k)

    # The retrieved passages get pasted straight into the prompt. This is
    # the entire "augmented" part of Retrieval-Augmented Generation.
    context = "\n\n".join(f"[{h['doc_id']}] {h['text']}" for h in hits)
    prompt = (
        "Answer the question using ONLY the context below. If the context "
        "doesn't contain the answer, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    sources = sorted({h["doc_id"] for h in hits})
    return call_ollama_chat(prompt), sources


print(f"QUESTION: {QUESTION}\n")

print("=" * 70)
print("RAG OFF - the model answers from memory alone")
print("=" * 70)
without_rag, _ = answer(QUESTION, use_rag=False)
print(without_rag.strip())

print("\n" + "=" * 70)
print("RAG ON - the same question, plus retrieved context")
print("=" * 70)
with_rag, sources = answer(QUESTION, use_rag=True)
print(with_rag.strip())
print(f"\ncontext pulled from: {', '.join(sources)}")

## F.1. Ask it something the corpus can't answer

Just as important as making RAG work is seeing it **decline**. The prompt in
`answer()` says to use *only* the provided context and to say so if the answer
isn't there.

A well-behaved RAG system answers "the context doesn't say." A poorly-behaved
one falls back on half-remembered training data and states something
confidently wrong — which is worse than useless, because it *looks* sourced.

In [ ]:
off_topic = "What is the boiling point of water at sea level?"

reply, sources = answer(off_topic, use_rag=True)

print(f"QUESTION: {off_topic}")
print(f"(Nothing in a Russian literature corpus answers this.)\n")
print(reply.strip())
print(f"\ncontext pulled from: {', '.join(sources)}")
print("\nRetrieval ALWAYS returns its top 4 chunks - there's no relevance")
print("threshold. So the model still got context; it was just useless context.")
print("Judging that is the model's job, and small models are inconsistent at it.")

## F.2. Your turn

Two things worth trying, both one-line edits:

1. **Change `QUESTION`** in Part E and re-run E and F. Try something the
   corpus covers well ("What is *Dead Souls* about?") and something it covers
   only in passing.

2. **Change `EMBED_MODEL`** in D.1 to `"mxbai-embed-large"`, then re-run D.1,
   E, and F. Different embedding models retrieve different passages — and
   different passages can produce a different final answer, from the same
   model and the same question.

The second one is the more interesting experiment: it shows that in a RAG
system, *retrieval quality* often matters more than the language model.

---

# Recap

The whole pipeline, in five steps:

| Step | What happens |
| --- | --- |
| **Chunk** | Documents cut into passages small enough to be relevant |
| **Embed** | Each passage becomes a vector — with the *document* prefix |
| **Retrieve** | Question becomes a vector — with the *query* prefix — then one dot product finds the closest passages |
| **Augment** | Those passages get pasted into the prompt as context |
| **Generate** | The model answers from the context, not from memory |

Two things worth carrying away:

**RAG doesn't teach the model anything.** The weights never change. You're
just doing a good job of handing it the right page at the right moment. That's
why RAG updates instantly when documents change, and why it can cite sources.

**It's the tool-use pattern again.** In the prompting notebook, the model
delegated arithmetic to a calculator. Here it delegates *recall* to a search
index. Same shape: notice what the model is bad at, and give it something that
is good at that instead.

Everything here ran through two Ollama endpoints — `/api/embed` and
`/api/generate` — called with the standard library. No vector database, no
framework, no LangChain. For a corpus this size, a NumPy matrix multiply *is*
the search engine.